In [ ]:
%cd ../../

In [ ]:
import torch
from dotenv import load_dotenv
from pathlib import Path
from src.model import EvolutionOperator
from src.data import CalixareneDataModule
from src.configs import TrainerArgs, CalixareneDataArgs # noqa
from torch_geometric.loader import DataLoader
from loguru import logger
# from dataclasses import asdict
# import linear_operator_learning as lol
import matplotlib.pyplot as plt
from src.data import mdtraj_load
import numpy as np
import os
from src.utils import compute_descriptors
import pandas as pd
# import wandb

# Assume there's a .env file on the root of the repo where "DATA_PATH" is defined
load_dotenv()

In [ ]:
from pathlib import Path
from src.checkpoints import resolve_checkpoint_path

checkpoint_source = "hf"  # set to "local" to use checkpoints you trained yourself
local_checkpoint_root = Path(".")

def checkpoint_path(repo_id, filename, repo_type="model"):
    if checkpoint_source == "local":
        return resolve_checkpoint_path(
            source="local",
            local_path=local_checkpoint_root / filename,
        )
    return resolve_checkpoint_path(
        source="hf",
        repo_id=repo_id,
        filename=filename,
        repo_type=repo_type,
    )


In [ ]:
ckpt_G2_path = checkpoint_path(repo_id="pnovelli/encoderops", filename="calixarene-G2/checkpoints/last.ckpt")
ckpt_G13_path = checkpoint_path(repo_id="pnovelli/encoderops", filename="calixarene-G1+3/checkpoints/last.ckpt")


In [ ]:
model_G2 = EvolutionOperator.load_from_checkpoint(ckpt_G2_path)
model_G13 = EvolutionOperator.load_from_checkpoint(ckpt_G13_path)
data_args = torch.load(ckpt_G2_path, weights_only=False)['CalixareneDataModule']['data_args']
data_args = CalixareneDataArgs(**data_args)
datamodule = CalixareneDataModule(model_G2.trainer_args, data_args = data_args, num_workers = 1)
datamodule.setup("fit")

In [ ]:
#[TODO]